<a href="https://colab.research.google.com/github/shrijav22/ML_LAB_2026-2027/blob/main/KNNalgorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



**K-Nearest Neighbors (KNN)**
 K-Nearest Neighbors (KNN) is one of the simplest and most intuitive machine learning algorithms. It's a supervised learning method that can be used for both classification and regression.

The core idea is simple: "You are defined by the company you keep." KNN assumes that similar data points exist in close proximity.

It's called a "lazy learner" (or instance-based learner) because it doesn't build a complex internal model during its "training" phase. Instead, it simply stores the entire training dataset. The real work (the computation) happens only when you ask it to make a prediction.

**How KNN Works: A Step-by-Step Guide**

Here is the step-by-step process for how KNN makes a prediction for a new, unseen data point:

Choose a 'k' value: You first decide how many "neighbors" to look at. This is the 'k' (e.g., k=3, k=5, k=10). This is the most important hyperparameter you need to set.

Calculate Distance: The algorithm takes the new data point and measures its distance to every single point in the training dataset. The most common distance metric is Euclidean distance (the straight-line "as the crow flies" distance).

Find the 'k' Neighbors: The algorithm identifies the top 'k' data points from the training set that are "closest" (have the smallest distance) to the new point.

Make a Prediction: This is where it splits based on the task:

For Classification (Voting): The new point is assigned the class that is most common among its 'k' neighbors. This is a simple "majority vote."

For Regression (Averaging): The new point is assigned the average value of its 'k' neighbors. (Sometimes a median is used to be more robust to outliers).


In [1]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 1. Load the dataset
data = pd.read_csv("netflix_titles.csv")

# 2. Select Features and Target
X = data[['country', 'release_year', 'rating', 'listed_in']]
y = data['type']

# 3. Preprocess the data
numeric_features = ['release_year']
categorical_features = ['country', 'rating', 'listed_in']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# 4. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 5. Create and Train the KNN Classifier
knn = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

knn.fit(X_train, y_train)

# 6. Make Predictions
y_pred = knn.predict(X_test)

# 7. Evaluate the Model
accuracy = accuracy_score(y_test, y_pred)

print("--- Model Evaluation (k=5) ---")
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=['Movie', 'TV Show']
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 8. Predict on a Single New Sample
new_title = pd.DataFrame({
    'country': ['India'],
    'release_year': [2023],
    'rating': ['TV-MA'],
    'listed_in': ['Dramas']
})

prediction = knn.predict(new_title)

print("\n--- New Prediction ---")
print(f"Prediction for new title: {prediction[0]}")


--- Model Evaluation (k=5) ---
Accuracy: 0.9467

Classification Report:
              precision    recall  f1-score   support

       Movie       0.95      0.97      0.96      1840
     TV Show       0.94      0.89      0.91       803

    accuracy                           0.95      2643
   macro avg       0.94      0.93      0.94      2643
weighted avg       0.95      0.95      0.95      2643


Confusion Matrix:
[[1791   49]
 [  92  711]]

--- New Prediction ---
Prediction for new title: Movie
